**Como está o consumo médio de energia dos clientes, por região?**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

SELECT * FROM 

CREATE OR REPLACE TABLE workspace.silver.energy_bill AS
SELECT id, fk_local_unit, consumption, price
FROM solaria.public.energy_bill
WHERE consumption > 0 AND price > 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY consumption DESC) = 1;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

CREATE OR REPLACE TABLE workspace.gold.dim_regiao_consumo AS
SELECT DISTINCT
    a.id AS regiao_id,
    a.state AS estado,
    a.city AS cidade
FROM solaria.public.local_unit lu
JOIN solaria.public.address a ON a.id = lu.fk_address;

CREATE OR REPLACE TABLE workspace.gold.fato_consumo AS
SELECT
    eb.id AS energy_bill_id,
    eb.fk_local_unit,
    a.id AS regiao_id,
    eb.consumption,
    eb.price
FROM workspace.silver.energy_bill eb
JOIN solaria.public.local_unit lu ON lu.id = eb.fk_local_unit
JOIN solaria.public.address a ON a.id = lu.fk_address;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.kpi_consumo_por_regiao AS
SELECT
    dr.estado,
    dr.cidade,
    AVG(fc.consumption) AS consumo_medio,
    STDDEV(fc.consumption) AS desvio_consumo,
    COUNT(*) AS total_contas,
    RANK() OVER (ORDER BY AVG(fc.consumption) DESC) AS ranking_consumo
FROM workspace.gold.fato_consumo fc
JOIN workspace.gold.dim_regiao_consumo dr ON dr.regiao_id = fc.regiao_id
GROUP BY dr.estado, dr.cidade;

CREATE OR REPLACE TABLE workspace.gold.kpi_preco_energia_por_regiao AS
SELECT
    dr.estado,
    dr.cidade,
    AVG(fc.price) AS preco_medio
FROM workspace.gold.fato_consumo fc
JOIN workspace.gold.dim_regiao_consumo dr ON dr.regiao_id = fc.regiao_id
GROUP BY dr.estado, dr.cidade;